# Funções acessórias

Funções de qualidade de dados importadas do notebook [utils/data_quality_helpers](#notebook-3809218967970367):

* **`analisar_nulos()`** - Analisar valores nulos em múltiplas tabelas de forma eficiente
* **`comparar_nulos()`** - Comparar snapshots de nulos antes e depois de transformações
* **`analisar_duplicados_chave_logica()`** - Analisar duplicados por chave lógica
* **`analisar_datas_invalidas()`** - Verificar datas inválidas em colunas de data
* **`deduplicate_hybrid()`** - Remover duplicados usando estratégia HYBRID

✅ **Importação centralizada**: `%run ./utils/data_quality_helpers`

In [0]:
# Required imports for helper functions
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
def analisar_nulos(dfs_dict, return_snapshots=True):
    """
    Analisa valores nulos em múltiplas tabelas de forma eficiente.
    
    Parâmetros:
    -----------
    dfs_dict : dict
        Dicionário com nome_tabela -> DataFrame
    return_snapshots : bool
        Se True, retorna dicionário com métricas detalhadas
    
    Retorna:
    --------
    dict : Dicionário com métricas de nulos por tabela (se return_snapshots=True)
    """
    
    snapshots = {}
    
    for table_name, df in dfs_dict.items():
        print(f"--- Tabela: {table_name.upper()} ---")
        
        # Cache das colunas para evitar múltiplas chamadas Analyze RPC
        cols = df.columns
        total_rows = df.count()
        
        # Contagem eficiente: uma única passagem pelos dados
        null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols]
        null_counts_row = df.select(*null_exprs).collect()[0]
        
        # Calcular total de valores nulos
        total_nulls = sum([null_counts_row[c] for c in cols])
        
        # Mostrar apenas o total
        print(f"  Total de registos: {total_rows:,}")
        print(f"  Total de valores nulos: {total_nulls:,}")
        print()
        
        # Guardar snapshot para comparação
        if return_snapshots:
            snapshots[table_name] = {
                "total_rows": total_rows,
                "total_nulls": total_nulls,
                "columns_with_nulls": {c: null_counts_row[c] for c in cols if null_counts_row[c] > 0}
            }
    
    return snapshots if return_snapshots else None

In [0]:
def comparar_nulos(snapshot_before, snapshot_after, table_list):
    """
    Compara snapshots de nulos antes e depois de uma transformação.
    
    Parâmetros:
    -----------
    snapshot_before : dict
        Snapshot de nulos antes da transformação (retornado por analisar_nulos)
    snapshot_after : dict
        Snapshot de nulos depois da transformação (retornado por analisar_nulos)
    table_list : list
        Lista de nomes de tabelas a comparar
    
    Retorna:
    --------
    bool : True se houve perda de dados detectada, False caso contrário
    """
    
    print("=" * 70)
    print("COMPARAÇÃO: DADOS PERDIDOS DURANTE A TRANSFORMAÇÃO?")
    print("=" * 70 + "\n")
    
    data_loss_detected = False
    
    for table_name in table_list:
        before = snapshot_before[table_name]
        after = snapshot_after[table_name]
        
        nulls_gained = after["total_nulls"] - before["total_nulls"]
        
        print(f"Tabela {table_name.upper()}:")
        
        if nulls_gained > 0:
            print(f"  ⚠️  ATENÇÃO: Aumentaram {nulls_gained:,} valores nulos!")
            print(f"     Antes: {before['total_nulls']:,} nulos | Depois: {after['total_nulls']:,} nulos")
            data_loss_detected = True
            
            # Identificar quais colunas ganharam nulos
            print(f"\n  Colunas afetadas:")
            for col in after["columns_with_nulls"]:
                nulls_before = before["columns_with_nulls"].get(col, 0)
                nulls_after = after["columns_with_nulls"][col]
                
                if nulls_after > nulls_before:
                    print(f"    - {col}: {nulls_before:,} → {nulls_after:,} (+{nulls_after - nulls_before:,})")
        
        elif nulls_gained < 0:
            print(f"  ✅ Diminuíram {abs(nulls_gained):,} valores nulos (inesperado, mas positivo)")
        
        else:
            print(f"  ✅ Nenhuma perda de dados detectada")
            print(f"     Antes: {before['total_nulls']:,} nulos | Depois: {after['total_nulls']:,} nulos")
        
        print()
    
    if data_loss_detected:
        print("\n⚠️  CONCLUSÃO: Foram detetados valores que se tornaram nulos.")
        print("   Isto pode indicar conversões falhadas (ex: strings inválidas em colunas de data/número).")
        print("   Recomenda-se investigar as colunas afetadas antes de prosseguir.\n")
    else:
        print("\n✅ CONCLUSÃO: Não houve perda de dados durante a transformação.")
        print("   Todas as conversões de tipo foram bem-sucedidas.\n")
    
    return data_loss_detected

In [0]:
def analisar_datas_invalidas(df, table_name, date_cols, lower_bound="1900-01-01", upper_bound=None, show_results=True):
    """
    Analisa datas inválidas em múltiplas colunas de data.
    
    Parâmetros:
    -----------
    df : DataFrame
        DataFrame a analisar
    table_name : str
        Nome da tabela (para identificação no output)
    date_cols : list
        Lista de colunas de data a verificar
    lower_bound : str
        Data mínima aceitável (default: "1900-01-01")
    upper_bound : str, Column, or None
        Data máxima aceitável. Pode ser:
        - String (ex: "2026-06-01")
        - Column expression (ex: F.current_date())
        - None = sem limite superior (para datas que podem estar no futuro)
    show_results : bool
        Se True, mostra resultados com display()
    
    Retorna:
    --------
    DataFrame com resumo de datas inválidas por coluna
    """
    
    print("=" * 80)
    print(f"{table_name.upper()} - VERIFICAÇÃO DE DATAS IMPOSSÍVEIS (Profiling)")
    print("=" * 80 + "\n")
    
    # Converter lower_bound para date
    if isinstance(lower_bound, str):
        lower_bound_date = F.lit(lower_bound).cast("date")
    else:
        lower_bound_date = lower_bound
    
    print(f"Critérios de validação:")
    print(f"  Lower bound: {lower_bound if isinstance(lower_bound, str) else 'Column expression'}")
    
    # Converter upper_bound para date (pode ser string, Column, ou None)
    if upper_bound is not None:
        if isinstance(upper_bound, str):
            upper_bound_date = F.lit(upper_bound).cast("date")
            print(f"  Upper bound: {upper_bound}")
        else:
            # upper_bound já é uma Column expression (ex: F.current_date())
            upper_bound_date = upper_bound
            print(f"  Upper bound: Column expression (ex: current_date())")
    else:
        upper_bound_date = None
        print(f"  Upper bound: N/A (datas futuras são válidas)")
    
    print("\nAnálise de datas inválidas por coluna:\n")
    
    results = []
    
    for col_name in date_cols:
        # Verificar se coluna existe
        if col_name not in df.columns:
            print(f"⚠️  Coluna '{col_name}' não existe na tabela {table_name}.\n")
            continue
        
        # Contar valores não-nulos
        non_null_count = df.filter(F.col(col_name).isNotNull()).count()
        
        # Contar datas < lower_bound
        before_lower = df.filter(
            F.col(col_name).isNotNull() & 
            (F.col(col_name) < lower_bound_date)
        ).count()
        
        # Contar datas > upper_bound (apenas se upper_bound definido)
        if upper_bound_date is not None:
            after_upper = df.filter(
                F.col(col_name).isNotNull() & 
                (F.col(col_name) > upper_bound_date)
            ).count()
        else:
            after_upper = 0
        
        # Total de datas inválidas
        total_invalid = before_lower + after_upper
        invalid_pct = round((total_invalid / non_null_count) * 100, 2) if non_null_count > 0 else 0
        
        results.append((
            col_name,
            non_null_count,
            before_lower,
            after_upper,
            total_invalid,
            invalid_pct
        ))
        
        print(f"📅 {col_name}:")
        print(f"   Total não-nulos: {non_null_count:,}")
        print(f"   < {lower_bound if isinstance(lower_bound, str) else 'lower_bound'}: {before_lower:,}")
        if upper_bound_date is not None:
            print(f"   > upper_bound: {after_upper:,}")
        print(f"   ⚠️  Total inválidas: {total_invalid:,} ({invalid_pct}%)\n")
    
    # Criar DataFrame resumo
    if results:
        # Nome da coluna condicional: "futuro" se upper_bound existe, senão "N/A"
        upper_col_name = "futuro" if upper_bound is not None else "N/A"
        
        results_df = spark.createDataFrame(
            results,
            ["coluna", "não_nulos", f"antes_{lower_bound[:4] if isinstance(lower_bound, str) else 'min'}", upper_col_name, "total_inválidas", "pct_inválidas"]
        )
        
        if show_results:
            print("\n📊 Resumo Visual:")
            display(results_df)
    else:
        results_df = None
    
    print("\n" + "=" * 80)
    print(f"✅ Profiling de datas {table_name.upper()} concluído")
    print("=" * 80)
    
    return results_df

In [0]:
def deduplicate_hybrid(df, key_cols, df_demo, table_name="table"):
    """
    Remove duplicados usando estratégia HYBRID:
    1. Ordena por fda_dt DESC (mais recente primeiro)
    2. Usa completeness_score DESC como tiebreaker (mais completo em empates)
    3. Usa caseversion DESC como tiebreaker final
    
    Esta abordagem balanceia temporal correctness com data quality.
    
    Parâmetros:
    -----------
    df : DataFrame
        DataFrame a deduplicate
    key_cols : list
        Lista de colunas que formam a chave lógica (ex: ["primaryid", "caseid", "drug_seq"])
    df_demo : DataFrame
        DataFrame DEMO contendo primaryid, fda_dt, caseversion
    table_name : str
        Nome da tabela (para logging)
    
    Retorna:
    --------
    DataFrame deduplicated
    """
    
    print(f"--- Tabela: {table_name.upper()} ---")
    
    # 1. Join para obter fda_dt e caseversion
    df_with_dates = df.join(
        df_demo.select("primaryid", "fda_dt", "caseversion"),
        on="primaryid",
        how="left"
    )
    
    # 2. Calcular completeness score (número de campos não-nulos)
    # Importante: calcular sobre as colunas ORIGINAIS, não incluir fda_dt/caseversion temporários
    original_cols = df.columns
    non_null_counts = [F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in original_cols]
    df_scored = df_with_dates.withColumn("completeness_score", sum(non_null_counts))
    
    # 3. Contagem antes
    total_antes = df_scored.count()
    
    # 4. Window HYBRID: recency first, completeness as tiebreaker
    # IMPORTANT: Cast caseversion to INT to avoid string ordering bug ("10" < "2" as strings!)
    w = Window.partitionBy(*key_cols).orderBy(
        F.desc("fda_dt"),                           # 1. Mais recente primeiro
        F.desc("completeness_score"),               # 2. Mais completo (tiebreaker)
        F.desc(F.col("caseversion").cast("int")),  # 3. Versão mais recente (NUMERIC ordering!)
        F.desc("primaryid")                         # 4. Deterministic ordering
    )
    
    df_dedup = (
        df_scored
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn", "fda_dt", "caseversion", "completeness_score")  # Remover colunas temporárias
    )
    
    # 5. Contagem depois
    total_depois = df_dedup.count()
    
    print(f"Total antes: {total_antes:,}")
    print(f"Duplicados removidos: {total_antes - total_depois:,}")
    print(f"Total depois: {total_depois:,}")
    print(f"   Chave lógica: {key_cols}\n")
    
    return df_dedup

In [0]:
def analisar_duplicados_chave_logica(df, table_name, key_cols, show_results=True):
    """
    Analisa duplicados por chave lógica numa tabela.

    Parâmetros:
    df : DataFrame
        DataFrame a analisar.
    table_name : str
        Nome da tabela, usado apenas para identificação no output.
    key_cols : list
        Lista de colunas que compõem a chave lógica.
    show_results : bool
        Se True, mostra o resultado com display().

    Retorna:
    DataFrame com as combinações de chave lógica que aparecem mais do que uma vez.
    """

    print(f"--- Duplicados por chave lógica: {table_name.upper()} ---")

    # Confirma se todas as colunas da chave existem na tabela
    missing_cols = [c for c in key_cols if c not in df.columns]

    if missing_cols:
        print(f"Colunas em falta para esta análise: {missing_cols}\n")
        return None

    # Agrupa pela chave lógica e identifica combinações repetidas
    duplicate_keys_df = (
        df.groupBy(key_cols)
          .count()
          .filter(F.col("count") > 1)
          .orderBy(F.desc("count"))
    )

    total_duplicate_keys = duplicate_keys_df.count()

    print(f"Número de chaves lógicas com mais de um registo: {total_duplicate_keys}")

    if show_results:
        display(duplicate_keys_df.limit(5))

    return duplicate_keys_df

In [0]:
print("✅ Funções acessórias importadas corretamente!")
print("\nFunções Disponiveis:")
print("  - analisar_nulos()")
print("  - comparar_nulos()")
print("  - analisar_datas_invalidas()")
print("  - deduplicate_hybrid()")
print("  - analisar_duplicados_chave_logica()")